<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/calibrated-typed-decision/CalibratedTypedDecisionDatasetBuilder_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Calibrated-Typed-Decisions (v2): Ddataset Builder**
Builds on v1: GoEmotions rater soft labels, option descriptions, harder distractors, more data, SciCite, and a held-out PubMedQA test.


---



## Cell 1 — Install
Libraries for the teacher, the datasets, and the label-similarity model.

In [ ]:
!pip install -q "transformers>=4.48" accelerate bitsandbytes datasets huggingface_hub sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.7 MB/s eta 0:00:00


## Cell 2 — Google Drive and Hugging Face login
Everything is saved to Drive as it's produced. The login needs a WRITE token.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import notebook_login
notebook_login()

## Cell 3 — Settings
Every choice in one place. Run once with SMOKE = True first.

In [ ]:
import os, json, random, re, tarfile, time, urllib.request
import numpy as np
import pandas as pd
import torch

SMOKE = True          # True: tiny end-to-end test run. False: the real v2 build.

SEED        = 0
HF_REPO     = "m-lagnajit/calibrated-typed-decisions"
TEACHER     = "Qwen/Qwen2.5-7B-Instruct"
OUT_DIR     = "/content/drive/MyDrive/ctd_v2_smoke" if SMOKE else "/content/drive/MyDrive/ctd_v2"
DESC_PATH   = "/content/drive/MyDrive/ctd_v2_label_descriptions.json"   # shared by smoke and full runs
N_PER_SPLIT = ({"train": 16, "validation": 16, "test": 16} if SMOKE
               else {"train": 3000, "validation": 300, "test": 300})
HELDOUT_N   = 16 if SMOKE else None        # None = all 1,000 PubMedQA questions

NOUL_RATE        = 0.3     # fraction of rows turned into yes/no questions
MIN_OPTIONS      = 3
MAX_OPTIONS      = 8
HARD_SHARE       = 0.5     # share of distractors taken from the labels closest to the right answer
HARD_POOL        = 10      # how many closest labels count as "close"
BATCH_SIZE       = 8
MAX_STATE_CHARS  = 1200
HELDOUT_CHARS    = 2000    # PubMedQA abstracts are longer
MIN_LETTER_MASS  = 0.5
MIN_FIT_ROWS     = 30
HUMAN_PRIOR      = 0.1     # pseudo-votes per option when turning rater votes into a soft label
EMBED_MODEL      = "BAAI/bge-small-en-v1.5"
DATASET_LICENSE  = "cc-by-4.0"
os.makedirs(OUT_DIR, exist_ok=True)

SOURCES = [
    dict(name="clinc_oos", path="clinc/clinc_oos", config="plus",
         text="text", label="intent", train="train", eval="test",
         rename={"oos": "out of scope"},
         instructions="Which intent best describes this request to a virtual assistant?",
         noul="Is this request asking for: {label}?",
         domain="requests to a virtual assistant",
         credit="CLINC150 (Larson et al., 2019), CC-BY-3.0"),
    dict(name="go_emotions", path="google-research-datasets/go_emotions", config="simplified",
         text="text", label="labels", multi=True, id_col="id", train="train", eval="test",
         instructions="Which emotion does this comment express?",
         noul="Does this comment express {label}?",
         domain="Reddit comments",
         credit="GoEmotions (Demszky et al., 2020), Apache-2.0; soft labels from its raw per-rater annotations"),
    dict(name="banking77", path="mteb/banking77", config=None,
         text="text", label="label", label_name_col="label_text", train="train", eval="test",
         instructions="Which banking intent best describes this customer message?",
         noul="Is the customer's intent in this message: {label}?",
         domain="customer messages to a bank",
         credit="BANKING77 (Casanueva et al., 2020), PolyAI/banking77, CC-BY-4.0; loaded via mteb/banking77"),
    dict(name="massive_intent", path="mteb/amazon_massive_intent", config="en",
         text="text", label="label", train="train", eval="test",
         instructions="Which intent best describes this request to a voice assistant?",
         noul="Is this request to a voice assistant asking for: {label}?",
         domain="requests to a voice assistant",
         credit="MASSIVE (FitzGerald et al., 2022), English subset, CC-BY-4.0; "
                "loaded via mteb/amazon_massive_intent"),
    dict(name="amazon_polarity", path="mteb/amazon_polarity", config=None,
         text="text", label="label", label_name_col="label_text", train="train", eval="test",
         instructions="What is the sentiment of this product review?",
         noul="Is the sentiment of this product review {label}?",
         domain="product reviews",
         credit="Amazon Review Polarity (Zhang et al., 2015), Apache-2.0 per fancyzhx/amazon_polarity; "
                "loaded via mteb/amazon_polarity"),
    dict(name="scicite", path="allenai/scicite",          # path is only an identifier here
         url="https://s3-us-west-2.amazonaws.com/ai2-s2-research/scicite/scicite.tar.gz",
         files={"train": "scicite/train.jsonl", "eval": "scicite/test.jsonl"},
         text="string", label="label", id_col="_sid", train="train", eval="eval", n_train=2000,
         license="apache-2.0",   # verified: the authors' repository, which contains the data, is Apache-2.0
         instructions="Why does this sentence from a scientific paper cite another work?",
         noul="Does the citation in this sentence fit this label: {label}?",
         domain="citation sentences from scientific papers",
         credit="SciCite (Cohan et al., 2019), Apache-2.0 (authors' repository); "
                "downloaded from the authors' official link"),
]

# Held out: used ONLY as a final test. Never trained on, never used to fit a dial.
HELDOUT = dict(name="pubmedqa", path="qiaojin/PubMedQA", config="pqa_labeled", split="train",
               fallback_url="https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json",
               license="mit",
               credit="PubMedQA (Jin et al., 2019), expert-labelled PQA-L subset, MIT. "
                      "Held out: never used for training or calibration.")

PERMISSIVE = {"cc-by-4.0", "cc-by-3.0", "cc-by-2.0", "apache-2.0", "mit", "cc0-1.0"}
LICENSE_OVERRIDES = {                        # copy path -> ORIGINAL dataset's license
    "mteb/banking77":             "cc-by-4.0",   # PolyAI/banking77
    "mteb/amazon_massive_intent": "cc-by-4.0",   # alexa/massive
}

# Short label descriptions written by hand, in our own words. Labels without one here get a
# description written by the teacher from real training examples (Cell 7), saved for review.
HAND_DESCS = {
    "go_emotions": {
        "admiration": "finding someone or something impressive or worthy of respect",
        "amusement": "finding something funny or entertaining",
        "anger": "strong displeasure or rage at someone or something",
        "annoyance": "mild irritation or being bothered",
        "approval": "agreeing with or supporting something",
        "caring": "showing concern or kindness, wanting to help",
        "confusion": "not understanding something, being puzzled",
        "curiosity": "wanting to know or learn more",
        "desire": "wanting or wishing for something",
        "disappointment": "sadness that something fell short of hopes",
        "disapproval": "disagreeing with or rejecting something",
        "disgust": "revulsion at something unpleasant or offensive",
        "embarrassment": "feeling awkward, self-conscious, or ashamed",
        "excitement": "feeling energized and eager about something",
        "fear": "feeling scared, afraid, or threatened",
        "gratitude": "thankfulness or appreciation",
        "grief": "deep sorrow, often over a loss or death",
        "joy": "happiness, delight, or pleasure",
        "love": "strong affection or fondness",
        "nervousness": "anxiety, worry, or unease",
        "optimism": "hopefulness about the future",
        "pride": "satisfaction in an achievement by oneself or someone close",
        "realization": "suddenly becoming aware of or understanding something",
        "relief": "easing of worry or stress after something difficult",
        "remorse": "regret or guilt about something one did",
        "sadness": "feeling down, unhappy, or sorrowful",
        "surprise": "being caught off guard by something unexpected",
        "neutral": "no particular emotion is expressed",
    },
    "amazon_polarity": {
        "negative": "the reviewer is unhappy with the product",
        "positive": "the reviewer is happy with the product",
    },
    "scicite": {
        "background": "the cited work gives background or context",
        "method": "the paper uses a method, tool, or dataset from the cited work",
        "result": "the paper compares its results or findings with the cited work",
    },
    "clinc_oos": {
        "out of scope": "the request is outside what the assistant can handle",
    },
    "pubmedqa": {
        "yes": "the evidence in the abstract supports yes",
        "no": "the evidence in the abstract supports no",
        "maybe": "the evidence is mixed or inconclusive",
    },
}

## Cell 4 — License gate
Same rule as v1: only permissive licenses, checked against each ORIGINAL dataset.

In [ ]:
from huggingface_hub import dataset_info

def hub_license(path):
    try:
        cd = dataset_info(path).card_data
        lic = getattr(cd, "license", None) if cd is not None else None
    except Exception as e:
        return f"error:{type(e).__name__}"
    if isinstance(lic, list):
        lic = ",".join(lic)
    return (lic or "none").lower()

kept = []
print(f"{'source':40s} {'license':14s} kept?")
for src in SOURCES:
    if "url" in src:
        lic, note = src["license"], "  (checked by hand: authors' repository)"
    else:
        lic, note = LICENSE_OVERRIDES.get(src["path"], hub_license(src["path"])), ""
    src["license"] = lic
    ok = lic in PERMISSIVE
    print(f"{src['path']:40s} {lic:14s} {'YES' if ok else 'no - check manually'}{note}")
    if ok:
        kept.append(src)
SOURCES = kept
print(f"{HELDOUT['path']:40s} {HELDOUT['license']:14s} HELD OUT (test only)")

source                                   license        kept?
clinc/clinc_oos                          cc-by-3.0      YES
google-research-datasets/go_emotions     apache-2.0     YES
mteb/banking77                           cc-by-4.0      YES
mteb/amazon_massive_intent               cc-by-4.0      YES
mteb/amazon_polarity                     apache-2.0     YES
allenai/scicite                          apache-2.0     YES  (checked by hand: authors' repository)
qiaojin/PubMedQA                         mit            HELD OUT (test only)


## Cell 5 — Load sources and pick examples
Choose which examples each split uses. No questions yet: those need descriptions first.

In [ ]:
from datasets import load_dataset, Dataset

def humanize(s):
    return s.replace("_", " ").strip()

def hub_load(src, split):
    try:
        return load_dataset(src["path"], src["config"], split=split)
    except RuntimeError as e:
        if "scripts are no longer supported" not in str(e):
            raise
        print(f"  {src['name']}: old script dataset, loading the Parquet copy instead")
        return load_dataset(src["path"], src["config"] or "default", split=split,
                            revision="refs/convert/parquet")

def url_load(src):
    """Download the authors' official archive once, then read only the text and label fields."""
    cache = f"/content/src_cache/{src['name']}"
    os.makedirs(cache, exist_ok=True)
    tar_path = cache + ".tar.gz"
    if not os.path.exists(tar_path):
        urllib.request.urlretrieve(src["url"], tar_path)
    with tarfile.open(tar_path) as t:
        try:
            t.extractall(cache, filter="data")
        except TypeError:
            t.extractall(cache)
    out = {}
    for split, rel in src["files"].items():
        rows = []
        with open(os.path.join(cache, rel)) as f:
            for line in f:
                if not line.strip():
                    continue
                ex = json.loads(line)
                lab = ex.get(src["label"])
                if lab is None:
                    continue
                rows.append({src["text"]: ex.get(src["text"]) or "",
                             src["label"]: str(lab).strip().lower(),
                             "_sid": str(ex.get("unique_id") or ex.get("id") or "")})
        out[split] = Dataset.from_list(rows)
    return out

def load_source(src):
    if "url" in src:
        d = url_load(src)
        tr, ev = d[src["train"]], d[src["eval"]]
    else:
        tr, ev = hub_load(src, src["train"]), hub_load(src, src["eval"])
    feat = tr.features[src["label"]]
    src["_lab2id"] = None
    if src.get("multi"):
        names = feat.feature.names
    elif hasattr(feat, "names"):
        names = feat.names
    elif getattr(feat, "dtype", "") == "string":
        names = sorted(set(tr.unique(src["label"])) | set(ev.unique(src["label"])))
        src["_lab2id"] = {n: i for i, n in enumerate(names)}
    else:
        all_ids = {int(x) for ds in (tr, ev) for x in ds.unique(src["label"])}
        pairs = {}
        for ds in (tr, ev):
            for start in range(0, len(ds), 10_000):
                chunk = ds[start:start + 10_000]
                for lab, txt in zip(chunk[src["label"]], chunk[src["label_name_col"]]):
                    pairs.setdefault(int(lab), str(txt))
                if len(pairs) == len(all_ids):
                    break
            if len(pairs) == len(all_ids):
                break
        names = [pairs.get(i, f"label {i}") for i in range(max(all_ids) + 1)]
    rename = src.get("rename", {})
    return tr, ev, [rename.get(n, humanize(n)) for n in names]

def label_of(ex, src):
    lab = ex[src["label"]]
    if src.get("multi"):
        if len(lab) != 1:              # keep only single-label examples
            return None
        lab = lab[0]
    elif src.get("_lab2id"):
        lab = src["_lab2id"][lab]
    return int(lab)

def pick(ds, src, n, rng):
    """Pick n usable examples in a seeded random order."""
    order = list(range(len(ds)))
    rng.shuffle(order)
    out = []
    for i in order:
        ex = ds[i]
        lab = label_of(ex, src)
        text = (ex[src["text"]] or "").strip()
        if lab is None or not text:
            continue
        sid = str(ex[src["id_col"]]) if src.get("id_col") else ""
        out.append(dict(i=i, text=text[:MAX_STATE_CHARS], lab=lab, sid=sid))
        if len(out) == n:
            break
    return out

def examples_per_label(ds, src, n_labels, rng, k=3, max_scan=20000):
    """Up to k real TRAINING texts per label: material for writing label descriptions."""
    order = list(range(len(ds)))
    rng.shuffle(order)
    out = {}
    for i in order[:max_scan]:
        ex = ds[i]
        lab = label_of(ex, src)
        text = (ex[src["text"]] or "").strip()
        if lab is None or not text:
            continue
        if len(out.setdefault(lab, [])) < k:
            out[lab].append(text[:200])
        if len(out) == n_labels and all(len(v) >= k for v in out.values()):
            break
    return out

PICKED, EXAMPLES, LOADED = {}, {}, []
for src in SOURCES:
    try:
        tr, ev, names = load_source(src)
    except Exception as e:
        print(f"SKIP {src['name']}: {type(e).__name__}: {e}")
        continue
    LOADED.append(src)
    rng = random.Random(f"{SEED}-{src['name']}")
    n_train = N_PER_SPLIT["train"] if SMOKE else src.get("n_train", N_PER_SPLIT["train"])
    nv, nt = N_PER_SPLIT["validation"], N_PER_SPLIT["test"]
    train_rows = pick(tr, src, n_train, rng)
    ev_rows = pick(ev, src, nv + nt, rng)          # validation and test: disjoint halves
    PICKED[src["name"]] = dict(names=names, train=train_rows,
                               validation=ev_rows[:nv], test=ev_rows[nv:])
    hand = HAND_DESCS.get(src["name"], {})
    if any(n not in hand for n in names):          # only sources that need written descriptions
        EXAMPLES[src["name"]] = examples_per_label(
            tr, src, len(names), random.Random(f"{SEED}-{src['name']}-examples"))
    print(f"{src['name']:16s} labels={len(names):3d}  train={len(train_rows)}  "
          f"validation={nv if len(ev_rows) >= nv else len(ev_rows)}  test={len(ev_rows[nv:])}")

clinc_oos        labels=151  train=3000  validation=300  test=300
go_emotions      labels= 28  train=3000  validation=300  test=300
banking77        labels= 77  train=3000  validation=300  test=300


Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

massive_intent   labels= 60  train=3000  validation=300  test=300
amazon_polarity  labels=  2  train=3000  validation=300  test=300
scicite          labels=  3  train=2000  validation=300  test=300


## Cell 5b — GoEmotions: what the raters actually chose
One row per rater in the raw version; we count votes per comment.

In [ ]:
GE_VOTES, GE_RATERS = {}, {}
if "go_emotions" in PICKED:
    raw = load_dataset("google-research-datasets/go_emotions", "raw", split="train").to_pandas()
    meta = {"text", "id", "author", "subreddit", "link_id", "parent_id", "created_utc",
            "rater_id", "example_very_unclear"}
    emo_cols = [c for c in raw.columns if c not in meta]
    clear = raw[~raw["example_very_unclear"].astype(bool)]        # raters who gave real labels
    sums = clear.groupby("id")[emo_cols].sum()
    GE_RATERS = clear.groupby("id").size().to_dict()
    GE_VOTES = {cid: dict(zip(emo_cols, map(int, row))) for cid, row in zip(sums.index, sums.values)}

    needed = [r["sid"] for s in ("train", "validation", "test") for r in PICKED["go_emotions"][s]]
    found = [s for s in needed if s in GE_VOTES]
    print(f"{len(emo_cols)} emotion columns | {len(GE_VOTES):,} comments with votes")
    print(f"our GoEmotions rows found in the raw votes: {len(found)}/{len(needed)}")
    counts = pd.Series([GE_RATERS[s] for s in found]).value_counts().sort_index() if found else pd.Series(dtype=int)
    print("raters per comment (our rows):", {int(k): int(v) for k, v in counts.items()})

28 emotion columns | 58,011 comments with votes
our GoEmotions rows found in the raw votes: 3600/3600
raters per comment (our rows): {2: 43, 3: 2654, 4: 120, 5: 783}


## Cell 6 — Load the teacher
Same 4-bit teacher as v1. It's needed earlier now, to write label descriptions.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(TEACHER)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    TEACHER,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16),
    device_map="auto")
model.eval()

LETTER_IDS = []
for d in range(MAX_OPTIONS):
    L = chr(65 + d)
    ids = set()
    for s in (L, " " + L):
        enc = tok.encode(s, add_special_tokens=False)
        if len(enc) == 1:
            ids.add(enc[0])
    LETTER_IDS.append(sorted(ids))
print("letter token ids:", LETTER_IDS)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

letter token ids: [[32, 362], [33, 425], [34, 356], [35, 422], [36, 468], [37, 434], [38, 479], [39, 472]]


## Cell 7 — Label descriptions
Hand-written ones where we have them; the teacher writes the rest from real training examples.

In [ ]:
def clean_desc(s):
    s = s.strip().split("\n")[0].strip().strip('"\'').strip()
    s = re.sub(r"^(description|answer|label)\s*:\s*", "", s, flags=re.I).rstrip(".").strip()
    return " ".join(s.split()[:14])

def tidy_desc(s):
    """Clean a teacher-written description: no assistant boilerplate, no capital first letter."""
    s = re.sub(r"^(?:the\s+)?(?:virtual|voice)\s+assistant(?:'s)?\s+", "", s.strip(), flags=re.I)
    s = re.sub(r"^(?:queries|query|questions|question|inquiry|inquiries)\s+(?:about|for|on|regarding)\s+",
               "asking about ", s, flags=re.I)
    s = s.rstrip(".").strip()
    return s[:1].lower() + s[1:]

def render(source_name, label):
    d = DESCS.get(source_name, {}).get(label, "")
    return f"{label}: {d}" if d else label

def desc_prompt(src, label, examples):
    lines = "\n".join(f"- {e}" for e in examples) or "- (no examples available)"
    user = (f'Here are example {src["domain"]} with the label "{label}":\n{lines}\n\n'
            f"Describe in one short phrase (at most 12 words) what this label means. "
            f"Reply with the phrase only.")
    msgs = [{"role": "system", "content": "You write short, plain descriptions of category labels."},
            {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_batch(prompts, max_new_tokens=32):
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

DESCS = json.load(open(DESC_PATH)) if os.path.exists(DESC_PATH) else {}
todo = []
for src in LOADED:
    names, d = PICKED[src["name"]]["names"], DESCS.setdefault(src["name"], {})
    for n, text in HAND_DESCS.get(src["name"], {}).items():
        d[n] = text                                  # hand-written always wins
    todo += [(src, i, n) for i, n in enumerate(names) if not d.get(n)]
DESCS["pubmedqa"] = dict(HAND_DESCS["pubmedqa"])
print(f"{len(todo)} labels need a written description")

for s in range(0, len(todo), 16):
    chunk = todo[s:s + 16]
    prompts = [desc_prompt(src, n, EXAMPLES.get(src["name"], {}).get(i, [])) for src, i, n in chunk]
    for (src, i, n), text in zip(chunk, generate_batch(prompts)):
        DESCS[src["name"]][n] = clean_desc(text)
    with open(DESC_PATH, "w") as f:
        json.dump(DESCS, f, indent=1, ensure_ascii=False)
    print(f"  written {min(s + 16, len(todo))}/{len(todo)}")

for src in LOADED:                                   # tidy every teacher-written description
    hand = HAND_DESCS.get(src["name"], {})
    for n, d in DESCS[src["name"]].items():
        if n not in hand:
            DESCS[src["name"]][n] = tidy_desc(d)

with open(DESC_PATH, "w") as f:
    json.dump(DESCS, f, indent=1, ensure_ascii=False)
print("\nA sample (all of them are in", DESC_PATH, "- edit any you dislike, then rerun this cell):")
for src in LOADED:
    names = PICKED[src["name"]]["names"]
    for n in random.Random(1).sample(names, min(3, len(names))):
        print(f"  [{src['name']}] {render(src['name'], n)}")

0 labels need a written description

A sample (all of them are in /content/drive/MyDrive/ctd_v2_label_descriptions.json - edit any you dislike, then rerun this cell):
  [clinc_oos] do you have pets: asking about pet ownership and details
  [clinc_oos] restaurant suggestion: suggestions for restaurants
  [clinc_oos] order: place or arrange purchase of items
  [go_emotions] approval: agreeing with or supporting something
  [go_emotions] love: strong affection or fondness
  [go_emotions] neutral: no particular emotion is expressed
  [banking77] card payment wrong exchange rate: incorrect exchange rate applied to card payment
  [banking77] virtual card not working: virtual card functionality issues
  [banking77] cancel transfer: request to cancel or reverse a transfer
  [massive_intent] calendar remove: remove calendar events and related data
  [massive_intent] music settings: settings related to music playback and player
  [massive_intent] takeaway query: asking about takeaway and deliver

## Cell 8 — Hard distractors: which labels are close to each other
Embed every 'label: description' and find each label's nearest neighbours.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
POOLS = {}
for src in LOADED:
    names = PICKED[src["name"]]["names"]
    E = embedder.encode([render(src["name"], n) for n in names],
                        normalize_embeddings=True, convert_to_numpy=True)
    S = E @ E.T                                           # cosine similarity, every pair of labels
    POOLS[src["name"]] = {g: [int(j) for j in np.argsort(-S[g]) if j != g][:HARD_POOL]
                          for g in range(len(names))}
    g = random.Random(2).randrange(len(names))
    print(f"[{src['name']}] closest to '{names[g]}':",
          [names[j] for j in POOLS[src["name"]][g][:4]])
del embedder
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[clinc_oos] closest to 'whisper mode': ['cancel', 'change volume', 'change speed', 'greeting']
[go_emotions] closest to 'neutral': ['annoyance', 'sadness', 'disgust', 'embarrassment']
[banking77] closest to 'beneficiary not allowed': ['transfer not received by recipient', 'declined transfer', 'failed transfer', 'cancel transfer']
[massive_intent] closest to 'transport query': ['transport traffic', 'transport taxi', 'recommendation locations', 'social query']
[amazon_polarity] closest to 'negative': ['positive']
[scicite] closest to 'background': ['result', 'method']


## Cell 9 — Build every question, plus the held-out PubMedQA test
Like v1's question builder, now with descriptions and hard distractors.

In [ ]:
def make_question(src, names, gold, rng):
    """Choice or yes/no question. Half the distractors come from the labels closest to the answer."""
    pool = POOLS[src["name"]][gold]
    others = [j for j in range(len(names)) if j != gold]
    if rng.random() < NOUL_RATE:
        is_true = rng.random() < 0.5
        if is_true:
            cand = gold
        elif pool and rng.random() < HARD_SHARE:
            cand = rng.choice(pool)                          # a "no" that's genuinely close
        else:
            cand = rng.choice(others)
        d = DESCS.get(src["name"], {}).get(names[cand], "")
        label_text = f"{names[cand]} ({d})" if d else names[cand]
        if rng.random() < 0.5:
            display, perm = ["no", "yes"], [0, 1]
        else:
            display, perm = ["yes", "no"], [1, 0]
        return dict(type="noul", instructions=src["noul"].format(label=label_text),
                    options=["false", "true"], option_descriptions=["", ""],
                    candidate=names[cand], gold_index=int(is_true),
                    display=display, display_perm=perm)

    k = max(2, min(len(names), rng.randint(MIN_OPTIONS, MAX_OPTIONS)))
    n_hard = min(len(pool), round((k - 1) * HARD_SHARE))
    hard = rng.sample(pool, n_hard)
    rest = [j for j in others if j not in hard]
    opts = [gold] + hard + rng.sample(rest, k - 1 - n_hard)
    rng.shuffle(opts)
    options = [names[j] for j in opts]
    return dict(type="choice", instructions=src["instructions"], options=options,
                option_descriptions=[DESCS.get(src["name"], {}).get(n, "") for n in options],
                candidate="", gold_index=opts.index(gold),
                display=[render(src["name"], n) for n in options], display_perm=list(range(k)))

ALL = {"train": [], "validation": [], "test": [], "heldout": []}
for src in LOADED:
    P, names = PICKED[src["name"]], PICKED[src["name"]]["names"]
    rng = random.Random(f"{SEED}-{src['name']}-questions")
    for split in ["train", "validation", "test"]:
        for ex in P[split]:
            q = make_question(src, names, ex["lab"], rng)
            ALL[split].append(dict(id=f"{src['name']}-{split}-{ex['i']:06d}", source=src["path"],
                                   source_license=src["license"], source_id=ex["sid"],
                                   state=ex["text"], gold=q["options"][q["gold_index"]], **q))

def load_heldout():
    try:
        ds = load_dataset(HELDOUT["path"], HELDOUT["config"], split=HELDOUT["split"])
        items = []
        for ex in ds:
            ctx = ex["context"]
            parts = ctx["contexts"] if isinstance(ctx, dict) else ctx
            items.append(dict(pid=str(ex["pubid"]), question=ex["question"],
                              text=" ".join(parts), gold=str(ex["final_decision"]).strip().lower()))
        return items
    except Exception as e:
        print("Hugging Face copy failed, using the authors' GitHub file instead:", type(e).__name__, e)
        path = "/content/src_cache/ori_pqal.json"
        os.makedirs("/content/src_cache", exist_ok=True)
        if not os.path.exists(path):
            urllib.request.urlretrieve(HELDOUT["fallback_url"], path)
        raw = json.load(open(path))
        return [dict(pid=str(k), question=v["QUESTION"], text=" ".join(v["CONTEXTS"]),
                     gold=str(v["final_decision"]).strip().lower()) for k, v in raw.items()]

items = load_heldout()
if HELDOUT_N:
    items = random.Random(SEED).sample(items, min(HELDOUT_N, len(items)))
H_OPTS = ["yes", "no", "maybe"]
h_rng = random.Random(f"{SEED}-pubmedqa-questions")
for it in items:
    if it["gold"] not in H_OPTS:
        continue
    perm = list(range(3))
    h_rng.shuffle(perm)                                     # the teacher sees a random order
    ALL["heldout"].append(dict(
        id=f"pubmedqa-heldout-{it['pid']}", source=HELDOUT["path"], source_license=HELDOUT["license"],
        source_id=it["pid"], type="choice",
        instructions=f"Research question: {it['question']}\nBased on the abstract, which answer does "
                     f"the evidence support?",
        state=it["text"][:HELDOUT_CHARS], options=H_OPTS,
        option_descriptions=[DESCS["pubmedqa"][o] for o in H_OPTS], candidate="",
        gold=it["gold"], gold_index=H_OPTS.index(it["gold"]),
        display=[render("pubmedqa", H_OPTS[c]) for c in perm], display_perm=perm))

print("rows:", {s: len(v) for s, v in ALL.items()})
ex = next(r for r in ALL["train"] if r["type"] == "choice")
print("\nexample question as the teacher will see it:\n", ex["instructions"])
for d, t in enumerate(ex["display"]):
    print(f"  {chr(65 + d)}. {t}")

rows: {'train': 17000, 'validation': 1800, 'test': 1800, 'heldout': 1000}

example question as the teacher will see it:
 Which intent best describes this request to a virtual assistant?
  A. meeting schedule: asking about scheduled meetings and their timings
  B. rollover 401k: rollover of 401k funds
  C. taxes: asking about tax payments and obligations
  D. spending history: asking about past spending
  E. account blocked: account access or transaction functionality restricted
  F. restaurant suggestion: suggestions for restaurants


## Cell 10 — The teacher scores every question
Same one-pass scoring as v1, resumable. Held-out questions are scored only as a reference.

In [ ]:
SYSTEM = "You are a careful text classifier. Reply with only the letter of the single best option."

def build_prompt(r):
    block = "\n".join(f"{chr(65 + d)}. {t}" for d, t in enumerate(r["display"]))
    user = (f"{r['instructions']}\n\nText:\n{r['state']}\n\n"
            f"Options:\n{block}\n\nReply with only the letter.")
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def score_batch(rows):
    enc = tok([build_prompt(r) for r in rows], return_tensors="pt", padding=True).to(model.device)
    try:
        out = model(**enc, logits_to_keep=1)
    except TypeError:
        out = model(**enc)
    logp = torch.log_softmax(out.logits[:, -1, :].float(), dim=-1)
    results = []
    for b, r in enumerate(rows):
        n = len(r["display"])
        disp = torch.stack([torch.logsumexp(logp[b, LETTER_IDS[d]], dim=0) for d in range(n)]).cpu()
        canon = torch.empty_like(disp)
        canon[torch.tensor(r["display_perm"])] = disp
        results.append(dict(teacher_option_logprobs=[round(x, 5) for x in canon.tolist()],
                            teacher_letter_mass=round(disp.exp().sum().item(), 5)))
    return results

def read_valid(path):
    rows = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
        with open(path, "w") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return rows

def run_split(split):
    path = f"{OUT_DIR}/{split}.jsonl"
    done = {r["id"] for r in read_valid(path)}
    todo = [r for r in ALL[split] if r["id"] not in done]
    print(f"{split}: {len(done)} already done, {len(todo)} to score")
    t0 = time.time()
    with open(path, "a") as f:
        for s in range(0, len(todo), BATCH_SIZE):
            batch = todo[s:s + BATCH_SIZE]
            for r, extra in zip(batch, score_batch(batch)):
                f.write(json.dumps({**r, **extra}, ensure_ascii=False) + "\n")
            f.flush()
            if (s // BATCH_SIZE) % 25 == 0:
                rate = (s + len(batch)) / max(1e-9, time.time() - t0)
                print(f"  {split}: {len(done) + s + len(batch)}/{len(ALL[split])}  ({rate:.1f} rows/s)")
    print(f"  {split}: finished, {len(ALL[split])} rows")

for split in ["validation", "test", "heldout", "train"]:
    run_split(split)

validation: 0 already done, 1800 to score
  validation: 8/1800  (5.5 rows/s)
  validation: 208/1800  (5.6 rows/s)
  validation: 408/1800  (6.0 rows/s)
  validation: 608/1800  (6.0 rows/s)
  validation: 808/1800  (5.9 rows/s)
  validation: 1008/1800  (6.0 rows/s)
  validation: 1208/1800  (6.0 rows/s)
  validation: 1408/1800  (5.5 rows/s)
  validation: 1608/1800  (5.4 rows/s)
  validation: finished, 1800 rows
test: 0 already done, 1800 to score
  test: 8/1800  (6.0 rows/s)
  test: 208/1800  (6.2 rows/s)
  test: 408/1800  (6.1 rows/s)
  test: 608/1800  (6.1 rows/s)
  test: 808/1800  (6.0 rows/s)
  test: 1008/1800  (6.0 rows/s)
  test: 1208/1800  (6.0 rows/s)
  test: 1408/1800  (5.6 rows/s)
  test: 1608/1800  (5.4 rows/s)
  test: finished, 1800 rows
heldout: 0 already done, 1000 to score
  heldout: 8/1000  (2.1 rows/s)
  heldout: 208/1000  (1.8 rows/s)
  heldout: 408/1000  (1.8 rows/s)
  heldout: 608/1000  (1.8 rows/s)
  heldout: 808/1000  (1.8 rows/s)
  heldout: finished, 1000 rows
train:

## Cell 11 — Calibrate, then make the soft labels
Dials fit on validation (never on held-out). GoEmotions soft labels come from the raters.

In [ ]:
CURRENT = {s["path"] for s in LOADED} | {HELDOUT["path"]}

def load_split(split):
    rows = [r for r in read_valid(f"{OUT_DIR}/{split}.jsonl") if r["source"] in CURRENT]
    good = [r for r in rows if r["teacher_letter_mass"] >= MIN_LETTER_MASS]
    return good, len(rows) - len(good)

DATA, DROPPED = {}, {}
for split in ["train", "validation", "test", "heldout"]:
    DATA[split], DROPPED[split] = load_split(split)
print("dropped (teacher did not answer with a letter):", DROPPED)

def probs_at(r, T, b=0.0):
    z = np.array(r["teacher_option_logprobs"]) / T
    if r["type"] == "noul":
        z[1] += b
    z = z - z.max()
    p = np.exp(z)
    return p / p.sum()

def nll(rows, T, b=0.0):
    return float(np.mean([-np.log(probs_at(r, T, b)[r["gold_index"]] + 1e-12) for r in rows]))

def ece(rows, T, b=0.0, n_bins=10):
    P = [probs_at(r, T, b) for r in rows]
    conf = np.array([p.max() for p in P])
    corr = np.array([int(p.argmax() == r["gold_index"]) for p, r in zip(P, rows)])
    total = 0.0
    for k in range(n_bins):
        lo, hi = k / n_bins, (k + 1) / n_bins
        m = (conf >= lo) & ((conf < hi) if k < n_bins - 1 else (conf <= hi))
        if m.any():
            total += m.mean() * abs(corr[m].mean() - conf[m].mean())
    return float(total), float(corr.mean())

GRID   = np.exp(np.linspace(np.log(0.25), np.log(20.0), 200))
B_GRID = np.linspace(-8.0, 8.0, 321)

def fit_T(rows):
    return float(GRID[np.argmin([nll(rows, t) for t in GRID])])

def fit_T_and_shift(rows):
    gap = np.array([r["teacher_option_logprobs"][1] - r["teacher_option_logprobs"][0] for r in rows])
    y = np.array([r["gold_index"] for r in rows])
    with np.errstate(over="ignore"):
        z = gap[None, None, :] / GRID[:, None, None] + B_GRID[None, :, None]
        p = 1.0 / (1.0 + np.exp(-z))
    pen = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)).mean(axis=2)
    i, j = np.unravel_index(np.argmin(pen), pen.shape)
    return float(GRID[i]), float(B_GRID[j])

def fit_group(rows, qtype):
    return fit_T_and_shift(rows) if qtype == "noul" else (fit_T(rows), 0.0)

POOLED = {}
for qtype in ["choice", "noul"]:
    val = [r for r in DATA["validation"] if r["type"] == qtype]
    POOLED[qtype] = fit_group(val, qtype) if len(val) >= MIN_FIT_ROWS else (1.0, 0.0)

TEMPS, REPORT = {}, []
for g in sorted({(r["source"], r["type"]) for r in DATA["validation"]}):
    val  = [r for r in DATA["validation"] if (r["source"], r["type"]) == g]
    test = [r for r in DATA["test"] if (r["source"], r["type"]) == g]
    if len(val) >= MIN_FIT_ROWS:
        (T, b), how = fit_group(val, g[1]), "own"
    else:
        (T, b), how = POOLED[g[1]], ("pooled" if POOLED[g[1]] != (1.0, 0.0) else "none")
    if how == "own" and (T in (float(GRID[0]), float(GRID[-1])) or b in (float(B_GRID[0]), float(B_GRID[-1]))):
        print(f"WARNING {g}: best setting sits at the edge of the search range (T={T:.2f}, b={b:+.2f})")
    TEMPS[g] = (T, b)
    if not test:
        continue
    T_dial = fit_T(val) if how == "own" else T
    e_raw, acc_raw = ece(test, 1.0)
    e_dial, _ = ece(test, T_dial)
    e_cal, acc_cal = ece(test, T, b)
    REPORT.append(dict(source=g[0], type=g[1], n_val=len(val), n_test=len(test),
                       acc_raw=acc_raw, acc_cal=acc_cal, T=T, b=b, fit=how,
                       ece_raw=e_raw, ece_dial=e_dial, ece_cal=e_cal))
if SMOKE:
    print("SMOKE run: too few rows to fit real dials. That's expected.")

print("\nTeacher calibration on the test split")
print(pd.DataFrame([{"source": x["source"].split("/")[-1], "type": x["type"], "val": x["n_val"],
                     "test": x["n_test"], "acc raw": f"{x['acc_raw']:.1%}", "acc cal": f"{x['acc_cal']:.1%}",
                     "T": f"{x['T']:.2f}", "nudge": f"{x['b']:+.2f}", "fit": x["fit"],
                     "ECE raw": f"{x['ece_raw']:.3f}", "ECE cal": f"{x['ece_cal']:.3f}"}
                    for x in REPORT]).to_string(index=False))

def human_soft_label(r):
    """GoEmotions only: what the raters chose, plus a small prior so nothing is exactly 0 or 1."""
    v = GE_VOTES.get(r.get("source_id", ""))
    if v is None:
        return None
    if r["type"] == "noul":
        n = GE_RATERS.get(r["source_id"], 0)
        p = (v.get(r["candidate"], 0) + HUMAN_PRIOR) / (n + 2 * HUMAN_PRIOR)
        return [round(1 - p, 6), round(p, 6)]
    c = np.array([v.get(o, 0) for o in r["options"]], dtype=float) + HUMAN_PRIOR
    return [round(float(x), 6) for x in c / c.sum()]

GE_PATH = next((s["path"] for s in LOADED if s["name"] == "go_emotions"), None)
for split in DATA:
    for r in DATA[split]:
        if split == "heldout":
            T, b = 1.0, 0.0                       # reference only: no dial is ever fit on held-out
        else:
            T, b = TEMPS.get((r["source"], r["type"]), POOLED.get(r["type"], (1.0, 0.0)))
        r["teacher_temperature"] = round(T, 4)
        r["teacher_shift"] = round(b, 4)
        r["teacher_probs"] = [round(float(p), 6) for p in probs_at(r, T, b)]
        r["human_probs"] = human_soft_label(r) if r["source"] == GE_PATH else None
        if split == "heldout":
            r["soft_label"], r["soft_label_source"] = None, "none (held out)"
        elif r["human_probs"] is not None:
            r["soft_label"], r["soft_label_source"] = r["human_probs"], "human_raters"
        else:
            r["soft_label"], r["soft_label_source"] = r["teacher_probs"], "teacher"

# Before / after: how wrong was the teacher on GoEmotions, compared with the raters?
GE_DIAG = {}
ge = [r for s in ("validation", "test") for r in DATA[s] if r["source"] == GE_PATH and r["human_probs"]]
if ge:
    t = [np.array(r["teacher_probs"]) for r in ge]
    h = [np.array(r["human_probs"]) for r in ge]
    GE_DIAG = dict(
        n=len(ge),
        teacher_top_wrong=float(np.mean([ti.argmax() != r["gold_index"] for ti, r in zip(t, ge)])),
        raters_top_matches_gold=float(np.mean([hi.argmax() == r["gold_index"] for hi, r in zip(h, ge)])),
        mean_gap=float(np.mean([0.5 * np.abs(ti - hi).sum() for ti, hi in zip(t, h)])))
    print(f"\nGoEmotions check on {GE_DIAG['n']} validation+test questions:")
    print(f"  teacher's top pick disagrees with the human answer: {GE_DIAG['teacher_top_wrong']:.1%}")
    print(f"  raters' top pick matches the human answer:          {GE_DIAG['raters_top_matches_gold']:.1%}")
    print(f"  average gap between teacher and rater chances:      {GE_DIAG['mean_gap']:.3f}  "
          f"(0 = identical, 1 = completely different)")

h = DATA["heldout"]
if h:
    e_h, acc_h = ece(h, 1.0)
    print(f"\nHeld-out PubMedQA, teacher as-is (reference only): accuracy {acc_h:.1%}, ECE {e_h:.3f}, n={len(h)}")

dropped (teacher did not answer with a letter): {'train': 5, 'validation': 1, 'test': 3, 'heldout': 0}

Teacher calibration on the test split
               source   type  val  test acc raw acc cal    T nudge fit ECE raw ECE cal
              scicite choice  216   214   79.9%   79.9% 5.46 +0.00 own   0.190   0.030
              scicite   noul   84    86   67.4%   74.4% 4.89 +0.70 own   0.237   0.067
            clinc_oos choice  206   202   89.6%   89.6% 3.92 +0.00 own   0.096   0.067
            clinc_oos   noul   94    98   77.6%   86.7% 3.22 +2.00 own   0.211   0.038
          go_emotions choice  201   216   56.9%   56.9% 8.86 +0.00 own   0.398   0.088
          go_emotions   noul   99    84   61.9%   73.8% 9.46 +0.60 own   0.353   0.110
amazon_massive_intent choice  210   212   90.1%   90.1% 5.46 +0.00 own   0.089   0.028
amazon_massive_intent   noul   89    85   85.9%   87.1% 4.10 +1.35 own   0.142   0.092
      amazon_polarity choice  216   190   96.3%   96.3% 5.46 +0.00 own   0.

## Cell 12 — Look at the rows before publishing
One row from GoEmotions, SciCite, and the held-out set.

In [ ]:
def show(r):
    keys = ["source", "type", "instructions", "state", "options", "option_descriptions", "gold",
            "teacher_probs", "human_probs", "soft_label_source"]
    print(json.dumps({k: r.get(k) for k in keys}, indent=2, ensure_ascii=False)[:1600])
    print("-" * 60)

for name in [GE_PATH, "allenai/scicite"]:
    rows = [r for r in DATA["test"] if r["source"] == name and r["type"] == "choice"]
    if rows:
        show(random.Random(3).choice(rows))
if DATA["heldout"]:
    show(DATA["heldout"][0])

{
  "source": "google-research-datasets/go_emotions",
  "type": "choice",
  "instructions": "Which emotion does this comment express?",
  "state": "So kinda like an assistant? Interesting",
  "options": [
    "sadness",
    "pride",
    "realization",
    "curiosity",
    "surprise"
  ],
  "option_descriptions": [
    "feeling down, unhappy, or sorrowful",
    "satisfaction in an achievement by oneself or someone close",
    "suddenly becoming aware of or understanding something",
    "wanting to know or learn more",
    "being caught off guard by something unexpected"
  ],
  "gold": "curiosity",
  "teacher_probs": [
    0.034402,
    0.041332,
    0.206621,
    0.676313,
    0.041332
  ],
  "human_probs": [
    0.04,
    0.04,
    0.04,
    0.84,
    0.04
  ],
  "soft_label_source": "human_raters"
}
------------------------------------------------------------
{
  "source": "allenai/scicite",
  "type": "choice",
  "instructions": "Why does this sentence from a scientific paper cite ano

## Cell 13 — Publish v2 (and keep v1 reachable)
Tags the current Hub version as v1.0 first, then pushes v2 with its card.

In [ ]:
from datasets import Dataset, DatasetDict
from huggingface_hub import DatasetCard, HfApi

KEEP = ["id", "source", "source_license", "source_id", "type", "instructions", "state",
        "options", "option_descriptions", "candidate", "gold", "gold_index", "display_perm",
        "teacher_option_logprobs", "teacher_temperature", "teacher_shift", "teacher_probs",
        "teacher_letter_mass", "human_probs", "soft_label", "soft_label_source"]

# Fixed column types. Without this, a split where a column is empty on every row (held-out's
# human_probs and soft_label) gets the type "null" and can't be uploaded with the others.
from datasets import Features, Value
try:
    from datasets import List as ListOf          # newer versions of `datasets`
except ImportError:
    from datasets import Sequence as ListOf      # older versions

FEATURES = Features({
    "id": Value("string"), "source": Value("string"), "source_license": Value("string"),
    "source_id": Value("string"), "type": Value("string"), "instructions": Value("string"),
    "state": Value("string"), "options": ListOf(Value("string")),
    "option_descriptions": ListOf(Value("string")), "candidate": Value("string"),
    "gold": Value("string"), "gold_index": Value("int64"), "display_perm": ListOf(Value("int64")),
    "teacher_option_logprobs": ListOf(Value("float64")), "teacher_temperature": Value("float64"),
    "teacher_shift": Value("float64"), "teacher_probs": ListOf(Value("float64")),
    "teacher_letter_mass": Value("float64"), "human_probs": ListOf(Value("float64")),
    "soft_label": ListOf(Value("float64")), "soft_label_source": Value("string"),
})
assert set(FEATURES) == set(KEEP), "FEATURES and KEEP must list the same columns"

dd = DatasetDict({s: Dataset.from_list([{k: r.get(k) for k in KEEP} for r in rows], features=FEATURES)
                  for s, rows in DATA.items()})
print(dd)

table = "\n".join(
    f"| `{x['source']}` | {x['type']} | {x['n_test']} | {x['acc_raw']:.3f} | {x['acc_cal']:.3f} | "
    f"{x['T']:.2f} | {x['b']:+.2f} | {x['fit']} | {x['ece_raw']:.3f} | {x['ece_cal']:.3f} |" for x in REPORT)
source_list = "\n".join(f"- {s.get('credit', s['path'])}" for s in LOADED)
ge_line = (f"On {GE_DIAG['n']} GoEmotions validation and test questions, the teacher's top pick disagreed "
           f"with the human answer {GE_DIAG['teacher_top_wrong']:.1%} of the time, which is why GoEmotions "
           f"soft labels in v2 come from the raters instead." if GE_DIAG else "")

CARD = f"""
# calibrated-typed-decisions (v2)

Typed decisions (multiple choice and yes/no) over short English texts. Every row has a human
gold label and a calibrated soft label, plus the raw ingredients to rebuild the soft label
your own way. v1 is still available: `load_dataset("{HF_REPO}", revision="v1.0")`.

## What changed from v1

- **GoEmotions soft labels come from the human raters**, not the teacher. {ge_line}
- **Options carry short descriptions** (`card arrival: the customer is waiting for a card`).
  Some are hand-written; the rest were written by the teacher from real training examples.
- **Harder distractors:** about half the wrong options are the labels closest to the right
  answer (by embedding similarity of label and description), so questions aren't trivially easy.
- **More data:** 3,000 training questions per source (2,000 for SciCite), plus SciCite as a
  new source.
- **A held-out split** from PubMedQA (see below).

## Splits

| split | rows | use |
|---|---|---|
| train | {len(DATA['train'])} | training |
| validation | {len(DATA['validation'])} | choosing epochs, fitting dials |
| test | {len(DATA['test'])} | final evaluation on the same sources |
| heldout | {len(DATA['heldout'])} | **test only**: a new source and a new kind of decision |

**Do not train on `heldout`, and do not fit dials on it.** It's 1,000 expert-labelled PubMedQA
questions (yes / no / maybe from an abstract), there to measure how well a model decides on
a label set and domain it has never seen.

## Fields

| field | meaning |
|---|---|
| `type` | `choice` or `noul` (yes/no; options are always `["false", "true"]`) |
| `instructions`, `state` | the question, and the text being judged |
| `options`, `option_descriptions` | option labels in canonical order, and their descriptions |
| `candidate` | yes/no rows: the label being asked about |
| `gold`, `gold_index` | the human label from the source dataset |
| `soft_label` | **the recommended training target**: rater votes for GoEmotions, the calibrated teacher otherwise |
| `soft_label_source` | `human_raters`, `teacher`, or `none (held out)` |
| `human_probs` | GoEmotions only: rater votes per option, plus {HUMAN_PRIOR} pseudo-votes per option |
| `teacher_option_logprobs` | the teacher's raw log-probability of each option's answer letter |
| `teacher_temperature`, `teacher_shift` | the dial and nudge fit on validation for this row's group |
| `teacher_probs` | the calibrated teacher distribution |
| `teacher_letter_mass` | how much probability the teacher put on answer letters at all |
| `display_perm` | the option order the teacher saw |
| `source_id` | the row's id in its source dataset, where one exists |

## Teacher calibration on the test split

| source | type | n | acc raw | acc calibrated | T | shift | fit | ECE raw | ECE calibrated |
|---|---|---|---|---|---|---|---|---|---|
{table}

## Sources

{source_list}
- {HELDOUT['credit']}

## Limitations

- Teacher soft labels reflect one 4-bit 7B teacher's judgement, calibrated on average per group.
- GoEmotions rater soft labels come from 3 raters per comment (5 when they disagreed), so they are coarse.
- Teacher-written option descriptions were reviewed by sampling, not one by one.
- Distractor difficulty depends on the embedding model (`{EMBED_MODEL}`).
- English only. Gold labels inherit any noise in the source datasets.
"""

if SMOKE:
    print("SMOKE run: not publishing. Set SMOKE = False and rerun from Cell 3.")
else:
    api = HfApi()
    try:
        api.create_tag(HF_REPO, tag="v1.0", repo_type="dataset",
                       tag_message="v1: five sources, teacher soft labels")
        print("Tagged the current Hub version as v1.0")
    except Exception as e:
        print("v1.0 tag not created (it may already exist):", type(e).__name__)
    dd.push_to_hub(HF_REPO)
    card = DatasetCard.load(HF_REPO)
    card.data.license = DATASET_LICENSE
    card.data.language = ["en"]
    card.data.task_categories = ["text-classification"]
    card.data.tags = ["calibration", "soft-labels", "distillation", "typed-decisions"]
    card.data.pretty_name = "Calibrated Typed Decisions"
    card.text = CARD
    card.push_to_hub(HF_REPO)
    print(f"Published v2: https://huggingface.co/datasets/{HF_REPO}")

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass', 'human_probs', 'soft_label', 'soft_label_source'],
        num_rows: 16995
    })
    validation: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass', 'human_probs', 'soft_label', 'soft_label_source'],
        num_rows: 1799
    })
    test: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'displ

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


v1.0 tag not created (it may already exist): HfHubHTTPError


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp232huq1j.parquet    : 100%|#########9| 1.07MB / 1.07MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/7.29k [00:00<?, ?B/s]

Published v2: https://huggingface.co/datasets/m-lagnajit/calibrated-typed-decisions




---



---



# **Calibrated-Typed-Decisions (v2.1): Corrected Soft Labels**
Adds `corrected_soft_label` and `soft_label_corrected`, computed from existing columns. No GPU, no re-scoring.


---



## Cell 1 — Install
No GPU needed: this update is pure arithmetic on existing columns.

In [ ]:
!pip install -q datasets huggingface_hub

## Cell 2 — Google Drive and Hugging Face login
Drive holds your student results; the login needs a WRITE token.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import notebook_login
notebook_login()

## Cell 3 — Settings and load v2


In [ ]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict, Value
try:
    from datasets import List as ListOf          # newer versions of `datasets`
except ImportError:
    from datasets import Sequence as ListOf      # older versions
from huggingface_hub import HfApi, DatasetCard

PUBLISH     = True                  # True: tag v2.0, then upload v2.1
HF_REPO     = "m-lagnajit/calibrated-typed-decisions"
PREV_TAG    = "v2.0"                 # the version being replaced stays loadable under this tag
RESULTS_DIR = "/content/drive/MyDrive/ctd_student_v2"   # where the student notebook saved its tables
EXPECTED_E  = (1391, 13996)          # Run E changed 1,391 of 13,996 teacher-labelled training rows

ds = load_dataset(HF_REPO, revision="main")
print(ds)
assert "soft_label" in ds["train"].column_names, "expected v2 on the Hub"
assert "corrected_soft_label" not in ds["train"].column_names, "v2.1 columns already exist"
DATA = {split: list(ds[split]) for split in ds}

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass', 'human_probs', 'soft_label', 'soft_label_source'],
        num_rows: 16995
    })
    validation: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass', 'human_probs', 'soft_label', 'soft_label_source'],
        num_rows: 1799
    })
    test: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'displ

## Cell 4 — The corrected soft label
Run E's fix, stored as new columns. Every row is checked.

In [ ]:
def corrected(row):
    """If the right answer doesn't hold the teacher's largest value, swap it with that value.
    Returns (label, was_changed). Ties: if the right answer is tied for the top, nothing changes.
    Rater labels and held-out rows are left alone."""
    if row["soft_label"] is None:                      # held-out: no training target at all
        return None, False
    soft = list(row["soft_label"])
    if row["soft_label_source"] != "teacher":          # GoEmotions rater votes: honest human uncertainty
        return soft, False
    g, top = row["gold_index"], max(soft)
    if soft[g] == top:                                 # right answer already holds (or ties for) the top
        return soft, False
    k = soft.index(top)                                # first position holding the top value
    soft[k], soft[g] = soft[g], soft[k]
    return soft, True

def run_e_target(row):
    """Exactly what Run E trained on in the student notebook, recomputed here as a cross-check."""
    soft = np.array(row["soft_label"], dtype=float)
    soft = soft / soft.sum()
    if row["soft_label_source"] == "teacher":
        k, g = int(soft.argmax()), row["gold_index"]
        if k != g:
            soft[k], soft[g] = soft[g], soft[k]
    return soft

for split, rows in DATA.items():
    for r in rows:
        r["corrected_soft_label"], r["soft_label_corrected"] = corrected(r)

# Check every row, not a sample
for split, rows in DATA.items():
    for r in rows:
        c, flag = r["corrected_soft_label"], r["soft_label_corrected"]
        if r["soft_label"] is None:
            assert c is None and not flag
            continue
        assert sorted(c) == sorted(r["soft_label"]), "the teacher's numbers must be kept, only reordered"
        if r["soft_label_source"] == "teacher":
            g = r["gold_index"]
            assert c[g] == max(c), "the right answer must hold the largest value (ties allowed)"
            assert flag == (r["soft_label"][g] < max(r["soft_label"]))
        else:
            assert c == list(r["soft_label"]) and not flag, "rater labels must be unchanged"
        c_norm = np.array(c, dtype=float) / sum(c)
        assert np.allclose(c_norm, run_e_target(r), atol=1e-12), "must equal Run E's training target"
print("every row checked: values kept, right answer holds the top value, rater rows and held-out untouched")
print("every row's corrected_soft_label equals the target Run E trained on")

train_teacher = [r for r in DATA["train"] if r["soft_label_source"] == "teacher"]
by_position = sum(int(np.argmax(r["soft_label"])) != r["gold_index"] for r in train_teacher)
really_changed = sum(r["soft_label_corrected"] for r in train_teacher)
ties = by_position - really_changed
print(f"\ntraining rows Run E's printout counted: {by_position} of {len(train_teacher)}  |  "
      f"expected {EXPECTED_E[0]} of {EXPECTED_E[1]} -> "
      f"{'MATCH' if (by_position, len(train_teacher)) == EXPECTED_E else 'DIFFERENT, check before publishing'}")
print(f"of those, actually changed: {really_changed}; right answer already tied for the top: {ties}")

# How often the teacher's top pick was wrong, per source and question type (train + validation + test)
rates = (pd.DataFrame([{"source": r["source"].split("/")[-1], "type": r["type"],
                        "changed": r["soft_label_corrected"]}
                       for s in ("train", "validation", "test") for r in DATA[s]
                       if r["soft_label_source"] == "teacher"])
         .groupby(["source", "type"])["changed"].agg(["sum", "count"]).reset_index())
rates["share"] = rates["sum"] / rates["count"]
print(rates.to_string(index=False, formatters={"share": "{:.1%}".format}))

every row checked: values kept, right answer holds the top value, rater rows and held-out untouched
every row's corrected_soft_label equals the target Run E trained on

training rows Run E's printout counted: 1391 of 13996  |  expected 1391 of 13996 -> MATCH
of those, actually changed: 1378; right answer already tied for the top: 13
               source   type  sum  count share
amazon_massive_intent choice  220   2516  8.7%
amazon_massive_intent   noul  108   1076 10.0%
      amazon_polarity choice   93   2496  3.7%
      amazon_polarity   noul   36   1104  3.3%
            banking77 choice  284   2484 11.4%
            banking77   noul  146   1116 13.1%
            clinc_oos choice  123   2461  5.0%
            clinc_oos   noul  104   1139  9.1%
              scicite choice  408   1832 22.3%
              scicite   noul  205    768 26.7%


## Cell 5 — The updated dataset card
Adds the new columns, the correction rates, and your student results.

In [ ]:
def fmt(v):
    if isinstance(v, float):
        return "" if np.isnan(v) else f"{v:.3f}"
    return str(v)

def md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)

rate_table = md_table(rates.assign(share=rates["share"].map("{:.1%}".format))
                      .rename(columns={"sum": "rows changed", "count": "teacher-labelled rows"}))

# Held-out baselines: what you get without reading the abstract at all
gold_h = pd.Series([r["gold"] for r in DATA["heldout"]])
freq = gold_h.value_counts(normalize=True)
majority_acc = float(freq.iloc[0])
base_rate_ll = float(-(freq * np.log(freq)).sum())

results_md = ""
paths = {s: f"{RESULTS_DIR}/results_{s}.csv" for s in ("test", "heldout")}
if all(os.path.exists(p) for p in paths.values()):
    keep = ["model", "accuracy", "ECE", "log loss", "accuracy after dial", "ECE after dial", "log loss after dial"]
    t_test, t_held = (pd.read_csv(paths[s])[keep] for s in ("test", "heldout"))
    results_md = f"""
## Which training target to use

We trained one ModernBERT-base student per target on the train split (3 epochs, best epoch chosen
on validation log loss), then fitted a temperature dial per source and question type on validation.
**These are single training runs**; they'll be updated once repeated with more seeds.

| run | target |
|---|---|
| A | one-hot `gold` |
| B | `gold`, label-smoothed (0.1) |
| C | `soft_label` |
| D | `0.5 * one_hot(gold) + 0.5 * soft_label` |
| E | `corrected_soft_label` |

**Test split** (the six training sources):

{md_table(t_test)}

**Held-out PubMedQA** (never trained on; dials pooled across sources from validation):

{md_table(t_held)}

For reference on held-out: always answering the most common label scores {majority_acc:.3f} accuracy,
and always predicting the label frequencies gives a log loss of {base_rate_ll:.3f}.

What we take from this run:

- **On the training sources, the 50/50 blend (D) did best**: the highest accuracy, the lowest ECE and
  log loss, and it beat the one-hot baseline on accuracy in a paired bootstrap.
- **On the held-out domain, targets that keep the teacher's full shape transferred better**:
  `corrected_soft_label` (E) had the best accuracy, `soft_label` (C) the best calibration after a dial.
  Every student stayed close to the baselines above, so transfer to this domain is weak.
- **`soft_label` alone (C) inherits the teacher's accuracy** on teacher-labelled sources;
  `corrected_soft_label` removes that ceiling.
"""
else:
    print("Student results not found in", RESULTS_DIR, "- the card will skip that section.")

v21_section = f"""## What changed in v2.1

- **New column `corrected_soft_label`**: identical to `soft_label`, except on rows where the calibrated
  teacher's top pick disagreed with the human answer. There, the teacher's value on its top pick and its
  value on the right answer are swapped: the teacher's numbers are kept, but the right answer gets the
  largest one. If the right answer is already tied for the teacher's largest value, the row is
  unchanged. Rater-labelled GoEmotions rows are unchanged; held-out rows stay empty.
- **New column `soft_label_corrected`**: `true` on exactly the rows where that swap changed the values.
- Nothing else changed. `soft_label`, `teacher_probs` and every other value are exactly as in v2.0.

How often the teacher's top pick was wrong (train, validation and test):

{rate_table}

"""

card = DatasetCard.load(HF_REPO)
text = card.text

def insert_before(text, marker, new):
    if marker in text:
        return text.replace(marker, new + marker, 1)
    print(f"  marker not found, appending instead: {marker!r}")
    return text + "\n" + new

text = text.replace("# calibrated-typed-decisions (v2)", "# calibrated-typed-decisions (v2.1)", 1)
old_line = f'v1 is still available: `load_dataset("{HF_REPO}", revision="v1.0")`.'
new_line = (f'Earlier versions stay available: `load_dataset("{HF_REPO}", revision="v1.0")` '
            f'and `revision="{PREV_TAG}"`.')
text = text.replace(old_line, new_line, 1) if old_line in text else text
text = insert_before(text, "## What changed from v1", v21_section)
field_row = "| `soft_label_source` | `human_raters`, `teacher`, or `none (held out)` |"
new_rows = ("\n| `corrected_soft_label` | `soft_label` with the teacher's wrong top picks corrected (see v2.1 above) |"
            "\n| `soft_label_corrected` | `true` where `corrected_soft_label` differs from `soft_label` |")
text = text.replace(field_row, field_row + new_rows, 1) if field_row in text else text + new_rows
if results_md:
    text = insert_before(text, "## Teacher calibration on the test split", results_md.strip() + "\n\n")
NEW_CARD = text
print(NEW_CARD[:3500])
print("\n... (", len(NEW_CARD), "characters in total )")

Student results not found in /content/drive/MyDrive/ctd_student_v2 - the card will skip that section.

# calibrated-typed-decisions (v2.1)

Typed decisions (multiple choice and yes/no) over short English texts. Every row has a human
gold label and a calibrated soft label, plus the raw ingredients to rebuild the soft label
your own way. Earlier versions stay available: `load_dataset("m-lagnajit/calibrated-typed-decisions", revision="v1.0")` and `revision="v2.0"`.

## What changed in v2.1

- **New column `corrected_soft_label`**: identical to `soft_label`, except on rows where the calibrated
  teacher's top pick disagreed with the human answer. There, the teacher's value on its top pick and its
  value on the right answer are swapped: the teacher's numbers are kept, but the right answer gets the
  largest one. If the right answer is already tied for the teacher's largest value, the row is
  unchanged. Rater-labelled GoEmotions rows are unchanged; held-out rows stay empty.
- **New column 

## Cell 6 — Publish v2.1 (and keep v2.0 reachable)
Tags the current Hub version as v2.0 first, then uploads.

In [ ]:
FEATURES = ds["train"].features.copy()               # v2's exact column types...
FEATURES["corrected_soft_label"] = ListOf(Value("float64"))
FEATURES["soft_label_corrected"] = Value("bool")     # ...plus the two new columns
dd = DatasetDict({s: Dataset.from_list(rows, features=FEATURES) for s, rows in DATA.items()})
print(dd)

if not PUBLISH:
    print("\nDry run: nothing uploaded. Check the output above, set PUBLISH = True, and rerun Cells 3 to 6.")
else:
    api = HfApi()
    try:
        api.create_tag(HF_REPO, tag=PREV_TAG, repo_type="dataset",
                       tag_message="v2.0: rater labels for GoEmotions, descriptions, hard distractors, held-out")
        print(f"Tagged the current Hub version as {PREV_TAG}")
    except Exception as e:
        print(f"{PREV_TAG} tag not created (it may already exist):", type(e).__name__)
    dd.push_to_hub(HF_REPO)
    card = DatasetCard.load(HF_REPO)
    card.text = NEW_CARD
    card.push_to_hub(HF_REPO)
    check = load_dataset(HF_REPO, revision="main", split="heldout")
    print("main now has:", [c for c in check.column_names if "corrected" in c])
    print(f"Published v2.1: https://huggingface.co/datasets/{HF_REPO}")

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass', 'human_probs', 'soft_label', 'soft_label_source', 'corrected_soft_label', 'soft_label_corrected'],
        num_rows: 16995
    })
    validation: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass', 'human_probs', 'soft_label', 'soft_label_source', 'corrected_soft_label', 'soft_label_corrected'],
        num_rows: 1799
    })
    test: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'ins

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Tagged the current Hub version as v2.0


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Generating train split:   0%|          | 0/16995 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1799 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1797 [00:00<?, ? examples/s]

Generating heldout split:   0%|          | 0/1000 [00:00<?, ? examples/s]

main now has: ['corrected_soft_label', 'soft_label_corrected']
Published v2.1: https://huggingface.co/datasets/m-lagnajit/calibrated-typed-decisions
